# 04 — Feature Engineering and Multi-Horizon Model Preparation (Annual Data)

**Project:** AI-Based Climate Change Awareness Dashboard  **Target:** NASA GISTEMP global temperature anomaly (annual)

This notebook **replaces nothing**. `03_feature_engineering_climate_forecast.ipynb` is left untouched; this is a new, stricter version built from its review.

## What this notebook does
1. Loads the annual dataset and **verifies** it (years sorted, unique, consecutive, correct columns).
2. Builds a **small, justified** set of features (CO₂ level / forcing / smoothing / growth / spread, natural drivers, optional greenhouse gases).
3. Builds **direct multi-horizon targets**: for origin year `t` and horizon `h`, the target is the real anomaly in calendar year `t + h`, found by a *year lookup*, never by shifting rows.
4. Compares **naive baselines**, **Linear Regression** (baseline model) and **Ridge** (main model) with a leakage-safe, chronological **walk-forward** procedure.
5. Runs **feature ablations** (add one feature group at a time, compare on identical rows) and picks a compact feature set using development data only.
6. Provides a **scenario feature builder** that needs only the supplied CO₂ (and optional CH₄/N₂O) pathway.
7. Runs automated **leakage / alignment / protected-file checks** and saves outputs to new folders only.

## What this notebook does NOT claim
- It is **not a physical climate model**. It is a simple regression on a few drivers, with ~145 annual points.
- Scenario curves are **exploratory statistical approximations**, not official IPCC projections, and are **not validated** just because historical metrics look reasonable. The `h=0` model is a same-year regression, not a forecast made before the year begins.
- Aerosols and volcanic forcing are **not in the data** (Phase 1 limitation). Mid-20th-century cooling from aerosols will therefore be attributed to the wrong things; every counterfactual inherits this.
- The 280 ppm curve is a **CO₂-held-constant statistical counterfactual**, not a causal estimate of a world without human influence. No result is claimed until the cells below have actually been run.

## 1. Decisions and assumptions (read this before the code)

| Decision | Reason (plain English) |
|---|---|
| **Direct horizons** `h = 5, 10, 15, 20, 25, 30` (plus `h = 1` baseline) | Predicting 30 years by feeding a 1-year model its own output compounds errors. Each horizon gets its own model and its own target. |
| **Target by calendar year**, not `shift(-n)` | `shift(-5)` means "5 rows later". If a year were missing, that would silently be the wrong year. |
| **Training rows are "purged"** | A training example from origin `o` has its answer in year `o+h`. We only train on examples whose answer year is **≤ the first test origin**, so no training label is "from the future" of the test forecast. |
| **Walk-forward (expanding window)** | Train on the past, test on the next block of years, repeat. Never a random split (climate data are strongly time-dependent). |
| **Last ~20% of origins = hold-out** | Feature *selection* uses development folds only. The hold-out is reported separately and is **not** used to choose features or `alpha`. |
| **Feature-selection verdicts prioritize h=1, 5, 10** | Longer-horizon targets overlap and provide weak independent evidence. Results at 15–30 years are reported as exploratory, not pooled into a single feature-selection verdict. `h=0` is excluded from forecast feature selection. |
| **Linear Regression = baseline model, Ridge = main model** | Ridge shrinks coefficients, which helps when predictors overlap (CO₂ features are all strongly correlated). |
| **Scaling inside a `Pipeline`** | The scaler is fitted on the training rows of each fold only. |
| **No imputation** | Rows missing any needed feature are *excluded* (and counted), never filled. Early rows missing a rolling window are simply not eligible. |
| **Same rows for every comparison** | A feature set is only compared against its base on rows where **both** can be computed, so one model is not "better" just because it was tested on easier years. |
| **CH₄/N₂O: use the reconstructed annual table** | Keep the reconstruction values so the historical feature table has the intended gas coverage. Preserve `_was_filled` and `_is_synthetic` provenance flags as audit metadata, never as model predictors. These reconstructed values use historical anchors and later observations, so gas-based results are retrospective/exploratory and may contain future-informed interpolation. |
| **Temperature features only in the 1-year experiment** | Lagged temperature makes a model "follow its own past" and cannot be used for scenarios, where future temperature is unknown. |
| **`h = 0` (same-year regression) is separate** | It maps that year's CO₂ features to that same year's anomaly. It is not a forecast made before that year's temperature is known and must not be presented as a conventional forecast metric. It is used only as an exploratory scenario-response mapping. Set `INCLUDE_CONTEMPORANEOUS = False` to remove it. |
| **Volcanic/aerosol forcing excluded** | No suitable complete global series. Stated limitation, not hidden. |
| **Sea level, sea ice, other temperature series, emissions** | Excluded: they are consequences/alternative measures of the target, or need a carbon-cycle model. |

In [ ]:
from pathlib import Path
import hashlib, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

SEED = 42                      # used by every random component (only the optional forest uses randomness)
np.random.seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# ------------------------------------------------------------------ configuration
HORIZONS = [5, 10, 15, 20, 25, 30]          # direct multi-horizon set (years ahead)
INCLUDE_ONE_YEAR = True                     # keep the h=1 baseline
INCLUDE_CONTEMPORANEOUS = True              # h=0 scenario-response model (see decisions table)
EVAL_HORIZONS = ([0] if INCLUDE_CONTEMPORANEOUS else []) + ([1] if INCLUDE_ONE_YEAR else []) + HORIZONS

CO2_PREINDUSTRIAL = 280.0                   # ppm, reference level for the forcing proxy
MIN_TRAIN = 25                              # minimum training rows for any fold
TEST_BLOCK = 5                              # origin years per development test block
HOLDOUT_FRAC = 0.20                         # share of latest eligible origins kept as hold-out
INNER_BLOCKS = 3                            # validation blocks used to choose Ridge alpha (training data only)
MIN_INNER_TRAIN = 15
ALPHAS = np.logspace(-3, 3, 7)              # Ridge regularisation grid
MIN_R2_N = 10                               # R2 is not reported for fewer test points than this
IMPROVE_MARGIN = 0.02                       # descriptive threshold, not a statistical significance test
MIN_TEST_FOR_VERDICT = 10                   # minimum pooled dev test origins; small samples remain weak evidence
VERDICT_HORIZONS = [1, 5, 10]               # forecast feature selection only; h=0 and 15-30y are reported separately
RUN_TREE_MODEL = False                      # optional RandomForest on the final compact set only
DRIVER_OVERLAP_TOL_PPM = 1.0                # max allowed CO2 disagreement with drivers_annual_full.csv

print("Evaluation horizons:", EVAL_HORIZONS)

In [ ]:
# ------------------------------------------------------------------ paths (relative to the repository root)
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "processed").is_dir():
            return p
    raise FileNotFoundError("Could not find data/processed above the current directory. "
                            "Run from the repository root or from notebooks/.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DATA_DIR = DATA_DIR / "multi_horizon_features"                          # NEW folder
OUT_REPORT_DIR = PROJECT_ROOT / "reports" / "feature_engineering" / "multi_horizon"   # NEW folder
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Fingerprint every existing processed CSV now; we re-check at the end that none changed.
INPUT_HASHES_BEFORE = {p.name: sha256(p) for p in sorted(DATA_DIR.glob("*.csv"))}

def safe_write_csv(df, path, **kw):
    """Only ever write inside the two NEW output folders. Refuses anything else."""
    path = Path(path).resolve()
    allowed = {OUT_DATA_DIR.resolve(), OUT_REPORT_DIR.resolve()}
    if path.parent not in allowed:
        raise PermissionError(f"Refusing to write outside the new output folders: {path}")
    df.to_csv(path, index=False, **kw)

def safe_write_text(text, path):
    path = Path(path).resolve()
    if path.parent not in {OUT_DATA_DIR.resolve(), OUT_REPORT_DIR.resolve()}:
        raise PermissionError(f"Refusing to write outside the new output folders: {path}")
    path.write_text(text, encoding="utf-8")

print("Project root:", PROJECT_ROOT)
print("Existing processed files fingerprinted:", list(INPUT_HASHES_BEFORE))

## 2. Load the reconstructed annual data (required)

This notebook deliberately loads `climate_annual_reconstructed.csv`, **not** `climate_annual.csv`. The reconstructed table preserves the original annual climate data and fills available CH₄/N₂O gaps using the reconstruction pipeline. We keep those reconstructed gas concentrations as candidate features and preserve their provenance flags for auditing. Provenance flags are **not** predictors. Because interpolation can use later anchors, gas-feature experiments are retrospective/exploratory, not strict real-time forecasts.


In [ ]:
def parse_bool_series(s, name):
    """Parse boolean flags explicitly; never use bool('False')."""
    if pd.api.types.is_bool_dtype(s):
        if s.isna().any():
            raise ValueError(f"Provenance flag {name!r} contains missing values.")
        return s.astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False,
               "yes": True, "no": False}
    parsed = s.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        bad = s[parsed.isna()].unique()[:5]
        raise ValueError(f"Column {name!r} has values that are not clearly boolean: {list(bad)}")
    return parsed.astype(bool)

annual_path = DATA_DIR / "climate_annual_reconstructed.csv"
if not annual_path.exists():
    raise FileNotFoundError(
        f"Required reconstructed input not found: {annual_path}. "
        "Run the approved GHG reconstruction pipeline first; this notebook will not silently use climate_annual.csv."
    )
USING_RECONSTRUCTED = True
raw = pd.read_csv(annual_path)
raw.columns = [c.strip() for c in raw.columns]
print("Loaded REQUIRED reconstructed input:", annual_path.name, "shape", raw.shape)

REQUIRED = ["year", "temp_anomaly", "co2", "ch4", "n2o"]
OPTIONAL = ["sunspots", "nino34"]
missing_required = [c for c in REQUIRED if c not in raw.columns]
if missing_required:
    raise ValueError(f"Reconstructed input is missing required columns: {missing_required}")
absent_optional = [c for c in OPTIONAL if c not in raw.columns]
if absent_optional:
    print("Optional columns not present (matching features will be skipped):", absent_optional)

# Strict numeric conversion: invalid text raises instead of silently becoming NaN.
raw["year"] = pd.to_numeric(raw["year"], errors="raise")
if not (raw["year"] % 1 == 0).all():
    raise ValueError("Non-integer years found.")
raw["year"] = raw["year"].astype(int)
for c in [c for c in REQUIRED[1:] + OPTIONAL if c in raw.columns]:
    raw[c] = pd.to_numeric(raw[c], errors="raise")

# The reconstruction pipeline should provide these flags. Require them so we can
# audit the reconstructed values instead of guessing which rows were filled.
PROVENANCE_COLUMNS = []
for gas in ["ch4", "n2o"]:
    for suffix in ["_was_filled", "_is_synthetic"]:
        col = gas + suffix
        if col not in raw.columns:
            raise ValueError(
                f"Missing required provenance column {col!r} in {annual_path.name}. "
                "Do not continue until the reconstructed dataset and its audit flags are available."
            )
        raw[col] = parse_bool_series(raw[col], col)
        PROVENANCE_COLUMNS.append(col)

GAS_RECONSTRUCTION_REPORT = {}
for gas in ["ch4", "n2o"]:
    filled = raw[gas + "_was_filled"]
    synthetic = raw[gas + "_is_synthetic"]
    GAS_RECONSTRUCTION_REPORT[gas] = {
        "non_missing_values": int(raw[gas].notna().sum()),
        "filled_values": int((filled & raw[gas].notna()).sum()),
        "synthetic_source_values": int((synthetic & raw[gas].notna()).sum()),
        "missing_values_remaining": int(raw[gas].isna().sum()),
    }
    # Safety checks: filled values must actually have a concentration, and no
    # row may be labelled filled while the reconstructed concentration is absent.
    assert not (filled & raw[gas].isna()).any(), f"{gas}: filled flag is True but concentration is missing"
    assert not (synthetic & raw[gas].isna()).any(), f"{gas}: synthetic flag is True but concentration is missing"

print("Reconstructed gas coverage (values retained; flags are audit metadata, not predictors):")
display(pd.DataFrame(GAS_RECONSTRUCTION_REPORT).T)
display(raw.head())
print("\nDtypes:\n", raw.dtypes.to_string())
print("\nMissing values per column:\n", raw.isna().sum().to_string())
if "co2_source" in raw.columns:
    print("\nco2_source values:\n", raw["co2_source"].value_counts(dropna=False).to_string())


In [ ]:
# ------------------------------------------------------------------ time-index validation
def validate_annual_index(df):
    """Hard checks. Returns the list of missing calendar years (if any)."""
    assert pd.api.types.is_integer_dtype(df["year"]), "year must be integer"
    assert not df["year"].duplicated().any(), "duplicate years found"
    assert df["year"].is_monotonic_increasing, "years are not sorted"
    return sorted(set(range(int(df["year"].min()), int(df["year"].max()) + 1)) - set(df["year"]))

was_sorted = raw["year"].is_monotonic_increasing
annual = raw.sort_values("year").reset_index(drop=True)
print("Input was sorted:", was_sorted)
GAP_YEARS = validate_annual_index(annual)
if GAP_YEARS:
    print("Missing calendar years inserted as NaN rows (not skipped!):", GAP_YEARS)
    y0, y1 = int(annual["year"].min()), int(annual["year"].max())
    annual = (annual.set_index("year").reindex(range(y0, y1 + 1))
                    .rename_axis("year").reset_index())
    # Newly inserted calendar rows have no gas observation; their provenance
    # flags are False (not missing/True), while the gas concentrations stay NaN.
    for flag in [c for c in annual.columns if c.endswith("_was_filled") or c.endswith("_is_synthetic")]:
        annual[flag] = annual[flag].fillna(False).astype(bool)
assert annual["year"].diff().dropna().eq(1).all(), "annual index is not consecutive"

print(f"Annual rows: {len(annual)}  | years {annual['year'].min()}–{annual['year'].max()}  | gap years: {GAP_YEARS or 'none'}")

coverage = pd.DataFrame({
    "first_valid_year": [annual.loc[annual[c].notna(), "year"].min() for c in annual.columns if c not in ("year", "co2_source")
                         and pd.api.types.is_numeric_dtype(annual[c])],
    "last_valid_year": [annual.loc[annual[c].notna(), "year"].max() for c in annual.columns if c not in ("year", "co2_source")
                        and pd.api.types.is_numeric_dtype(annual[c])],
    "n_valid": [int(annual[c].notna().sum()) for c in annual.columns if c not in ("year", "co2_source")
                and pd.api.types.is_numeric_dtype(annual[c])],
}, index=[c for c in annual.columns if c not in ("year", "co2_source") and pd.api.types.is_numeric_dtype(annual[c])])
display(coverage)
print("Coverage tells us which features can only be tested on part of the record (e.g. gases, Niño 3.4).")

In [ ]:
# ------------------------------------------------------------------ optional longer CO2 history (rolling windows only)
def load_co2_prehistory(annual_df):
    """Use drivers_annual_full.csv ONLY to give early model years a full CO2 rolling window.
    It is accepted only if it matches our annual CO2 where the two overlap."""
    path = DATA_DIR / "drivers_annual_full.csv"
    if not path.exists():
        return None, "drivers_annual_full.csv not found - early rolling CO2 features will be missing (rows excluded)"
    d = pd.read_csv(path)
    d.columns = [c.strip().lower() for c in d.columns]
    if not {"year", "co2"} <= set(d.columns):
        return None, "drivers_annual_full.csv has no year/co2 columns - ignored"
    d = d[["year", "co2"]].dropna()
    d["year"] = pd.to_numeric(d["year"], errors="raise").astype(int)
    d["co2"] = pd.to_numeric(d["co2"], errors="raise")
    d = d.drop_duplicates("year").sort_values("year").set_index("year")["co2"]
    y0 = int(annual_df["year"].min())
    overlap = annual_df.set_index("year")["co2"].dropna().index.intersection(d.index)
    if len(overlap) < 10:
        return None, "drivers file does not overlap enough with the annual table - ignored"
    max_diff = float((annual_df.set_index("year").loc[overlap, "co2"] - d.loc[overlap]).abs().max())
    if max_diff > DRIVER_OVERLAP_TOL_PPM:
        return None, f"drivers CO2 disagrees with annual CO2 by up to {max_diff:.2f} ppm - ignored"
    pre = d[d.index < y0]
    if pre.empty or pre.index.max() != y0 - 1:
        return None, "drivers file has no contiguous history ending the year before the annual table - ignored"
    # keep only the contiguous block ending at y0-1
    ys = list(pre.index)
    start = ys[-1]
    for y in reversed(ys[:-1]):
        if y == start - 1:
            start = y
        else:
            break
    pre = pre.loc[start:y0 - 1]
    return pre, f"using {len(pre)} pre-{y0} CO2 values ({pre.index.min()}–{pre.index.max()}) for rolling windows only"

CO2_PREHISTORY, prehistory_msg = load_co2_prehistory(annual)
print(prehistory_msg)

## 3. Build candidate features

All features are computed from information available **at the forecast origin year `t`** (the current year and earlier). Rolling windows are **trailing** (`center=False` is the pandas default; we also test it below by truncating the data).

| Idea | What it measures |
|---|---|
| **Rolling mean** (`co2_roll5`, `co2_roll10`, `sunspots_roll11`) | the *typical level* over the window — smooths noise, mimics slow (ocean) response |
| **Growth** (`co2_growth_1y`, `co2_growth_roll5`, `co2_forcing_change_5y`) | how *fast* the value is changing |
| **Rolling standard deviation** (`co2_std5`) | how much values *spread around their window mean*. For a smooth, steadily rising CO₂ curve this mostly re-encodes the growth rate, so it must **earn its place** in the ablation |

**Deliberately not built** (reason in the registry below): 3-year temperature mean (exact combination of temperature, lag 1 and lag 2), rolling means/std of *every* variable, one-year changes of every variable (exact differences of existing columns), CH₄/N₂O lags, provenance flags as predictors (they mostly mark "before ~1984", which duplicates the time trend).

In [ ]:
CO2_FEATURES = ["co2", "ln_co2", "co2_forcing", "co2_roll5", "co2_roll10", "co2_growth_1y",
                "co2_growth_roll5", "co2_std5", "co2_forcing_change_5y"]
GAS_FEATURES = ["ch4", "n2o"]
NATURAL_FEATURES = ["sunspots", "sunspots_roll11", "nino34"]
TEMP_FEATURES = ["temp_at_origin", "temp_lag1", "temp_roll5"]          # one-year experiment ONLY
TIME_FEATURES = ["year_index"]                                          # one-year experiment ONLY
SCENARIO_SAFE = set(CO2_FEATURES) | set(GAS_FEATURES)                    # computable from a supplied CO2/CH4/N2O pathway

def _assert_consecutive(index, what):
    idx = pd.Index(index)
    assert idx.is_unique, f"{what}: duplicate years"
    assert idx.is_monotonic_increasing, f"{what}: years not sorted"
    assert (np.diff(idx.to_numpy()) == 1).all(), f"{what}: years not consecutive"

def co2_feature_frame(co2):
    """The ONE place CO2 features are defined. Used for history AND for scenarios,
    so both always use identical formulas. `co2` is a Series indexed by consecutive years."""
    _assert_consecutive(co2.index, "co2 series")
    co2 = co2.astype(float)
    f = pd.DataFrame(index=co2.index)
    f["co2"] = co2
    f["ln_co2"] = np.log(co2)
    f["co2_forcing"] = 5.35 * np.log(co2 / CO2_PREINDUSTRIAL)      # CO2-only radiative forcing PROXY (W/m^2)
    f["co2_roll5"] = co2.rolling(5, min_periods=5).mean()           # trailing 5y mean (t-4..t)
    f["co2_roll10"] = co2.rolling(10, min_periods=10).mean()        # trailing 10y mean (t-9..t)
    growth = co2.diff(1)                                            # ppm/yr, uses t and t-1
    f["co2_growth_1y"] = growth
    f["co2_growth_roll5"] = growth.rolling(5, min_periods=5).mean() # 5y-smoothed growth rate
    f["co2_std5"] = co2.rolling(5, min_periods=5).std(ddof=1)       # trailing 5y spread of the CO2 level
    f["co2_forcing_change_5y"] = f["co2_forcing"].diff(5)           # forcing(t) - forcing(t-5)
    return f

def build_features(annual_df, co2_prehistory=None):
    """Feature table with one row per forecast-origin year t. Pure function of its inputs
    (so we can test it by truncating the data)."""
    a = annual_df.set_index("year")
    _assert_consecutive(a.index, "annual table")
    co2_full = a["co2"]
    if co2_prehistory is not None and len(co2_prehistory):
        co2_full = pd.concat([co2_prehistory, a["co2"]]).sort_index()
        _assert_consecutive(co2_full.index, "co2 with prehistory")
    out = co2_feature_frame(co2_full).loc[a.index].copy()

    out["temp_at_origin"] = a["temp_anomaly"]                              # year t (known at origin t)
    out["temp_lag1"] = a["temp_anomaly"].shift(1)                          # year t-1
    out["temp_roll5"] = a["temp_anomaly"].rolling(5, min_periods=5).mean() # years t-4..t
    if "sunspots" in a:
        out["sunspots"] = a["sunspots"]
        out["sunspots_roll11"] = a["sunspots"].rolling(11, min_periods=11).mean()   # trailing solar-cycle mean
    if "nino34" in a:
        out["nino34"] = a["nino34"]
    for gas in GAS_FEATURES:
        if gas in a:
            out[gas] = a[gas]  # reconstructed annual concentration; provenance retained below
        for suffix in ("_was_filled", "_is_synthetic"):
            flag = gas + suffix
            if flag in a:
                out[flag] = a[flag].astype(bool)  # audit metadata only; never passed as a predictor
    out["year_index"] = a.index - a.index.min()

    # Direct targets by CALENDAR-YEAR lookup (never shift by rows).
    temp = a["temp_anomaly"]
    for h in EVAL_HORIZONS:
        target_year = a.index + h
        out[f"target_year_h{h}"] = target_year
        out[f"target_h{h}"] = temp.reindex(target_year).to_numpy()
    out.index.name = "origin_year"
    return out.reset_index()

FEATURE_TABLE = build_features(annual, CO2_PREHISTORY)
PROVENANCE_COLUMNS = [c for c in FEATURE_TABLE.columns
                      if c.endswith("_was_filled") or c.endswith("_is_synthetic")]
ALL_FEATURES = [c for c in FEATURE_TABLE.columns
                if not c.startswith("target") and c not in (["origin_year"] + PROVENANCE_COLUMNS)]
print("Feature table:", FEATURE_TABLE.shape, "| features:", len(ALL_FEATURES))
display(FEATURE_TABLE[["origin_year", "co2", "co2_forcing", "co2_roll10", "co2_std5", "temp_at_origin",
                       "target_year_h5", "target_h5", "target_year_h30", "target_h30"]].tail(8))

## 4. Why the dataset is small (and why more columns do not help)

- **Rows = years.** One row per year means ~146 rows. Adding engineered columns adds *columns*, not *independent observations*.
- A **horizon-`h` target removes the last `h` origin years** (their answer lies beyond the data). So long horizons have **fewer** rows. The table below is the *actual* count.
- A rolling window of length `w` makes the first `w-1` rows unavailable (unless earlier history exists). We drop them; we never fill them with zeros.
- Monthly data would give ~12× more rows, but neighbouring months are almost copies of each other (autocorrelation), so the *independent information* hardly grows. It stays a separate secondary experiment.
- Generating synthetic rows would only repeat what the model already knows and would make test scores meaningless. We never do it.

In [ ]:
def eligible_rows(table, h, features):
    """Rows where the h-year target AND every requested feature exist and are finite. No imputation."""
    need = list(dict.fromkeys(list(features) + ["temp_at_origin", f"target_h{h}"]))
    vals = table[need].to_numpy(dtype=float)
    ok = np.isfinite(vals).all(axis=1)
    return table.loc[ok].sort_values("origin_year").reset_index(drop=True)

count_rows = []
for h in EVAL_HORIZONS:
    base_rows = eligible_rows(FEATURE_TABLE, h, ["co2_forcing"])
    full10 = eligible_rows(FEATURE_TABLE, h, ["co2_forcing", "co2_roll10"])
    nino = eligible_rows(FEATURE_TABLE, h, ["co2_forcing", "nino34"]) if "nino34" in FEATURE_TABLE else None
    ch4 = eligible_rows(FEATURE_TABLE, h, ["co2_forcing", "ch4"]) if "ch4" in FEATURE_TABLE else None
    count_rows.append({"horizon": h,
                       "rows_with_target": int(FEATURE_TABLE[f"target_h{h}"].notna().sum()),
                       "usable_CO2_forcing": len(base_rows),
                       "usable_with_co2_roll10": len(full10),
                       "usable_with_nino34": None if nino is None else len(nino),
                       "usable_with_ch4": None if ch4 is None else len(ch4)})
SAMPLE_COUNTS = pd.DataFrame(count_rows)
display(SAMPLE_COUNTS)
print("Rows shrink with horizon, and any feature that only exists later (gases, Niño 3.4) shrinks them further.")

## 5. Feature registry

One row per feature considered, including the ones we reject, so the design is defensible and auditable.

In [ ]:
REGISTRY = [
 # name, definition, motivation, scenario_capable, missing-data note, status/use
 ("co2", "annual CO2 (ppm) at year t", "main greenhouse-gas driver", True, "complete in annual table", "implemented; used in transformation test"),
 ("ln_co2", "ln(co2)", "warming responds roughly to log(CO2)", True, "none", "implemented; EXACT linear transform of co2_forcing -> identical predictions for linear models; diagnostic only"),
 ("co2_forcing", "5.35*ln(co2/280)", "CO2-only radiative-forcing PROXY (not total forcing)", True, "none", "implemented; BASE feature of every comparison"),
 ("co2_roll5", "mean(co2[t-4..t])", "slow (ocean-lagged) response", True, "first 4 rows unless pre-history available", "implemented; ablation"),
 ("co2_roll10", "mean(co2[t-9..t])", "slower response, planned in handoff", True, "first 9 rows unless pre-history available", "implemented; ablation"),
 ("co2_growth_1y", "co2[t]-co2[t-1]", "speed of increase (planned CO2 growth)", True, "first row", "implemented; ablation"),
 ("co2_growth_roll5", "mean of co2_growth_1y over t-4..t", "smoothed growth rate", True, "first 5 rows unless pre-history", "implemented; ablation"),
 ("co2_std5", "std(co2[t-4..t], ddof=1)", "user-requested; spread of CO2 in window", True, "first 4 rows unless pre-history", "implemented; ablation (suspected to duplicate growth)"),
 ("co2_forcing_change_5y", "forcing[t]-forcing[t-5]", "5-year forcing change", True, "first 5 rows unless pre-history", "implemented; ablation"),
 ("ch4", "reconstructed annual CH4 concentration (measured values retained; gaps filled where available)", "second greenhouse gas", True, "reconstructed values are estimates and may use later anchors; inspect provenance flags", "implemented; ablation; retrospective/exploratory"),
 ("n2o", "reconstructed annual N2O concentration (measured values retained; gaps filled where available)", "third greenhouse gas", True, "reconstructed values are estimates and may use later anchors; inspect provenance flags", "implemented; ablation; retrospective/exploratory"),
 ("sunspots", "annual sunspot number at t", "solar variability", False, "none; FUTURE values unknown in scenarios", "implemented; historical experiment only"),
 ("sunspots_roll11", "mean(sunspots[t-10..t])", "solar-cycle smoothing", False, "first 10 rows", "implemented; historical experiment only"),
 ("nino34", "annual Nino 3.4 at t", "ENSO (short-term variability)", False, "~half of record missing; future unknown", "implemented; historical experiment only"),
 ("temp_at_origin", "temp_anomaly[t]", "persistence of the anomaly", False, "none", "one-year experiment ONLY; banned when h=0 (it is the target)"),
 ("temp_lag1", "temp_anomaly[t-1]", "recent temperature memory", False, "first row", "one-year experiment ONLY"),
 ("temp_roll5", "mean(temp_anomaly[t-4..t])", "smoothed temperature level", False, "first 4 rows", "one-year experiment ONLY"),
 ("year_index", "year - first_year", "explicit time trend", False, "none", "one-year experiment ONLY; would make scenarios follow a straight line in time"),
 # rejected / not built
 ("temp_roll3", "mean(temp[t-2..t])", "-", False, "-", "NOT BUILT: exact average of temp_at_origin, temp_lag1, temp_lag2"),
 ("*_change_1y for every variable", "x[t]-x[t-1]", "-", False, "-", "NOT BUILT: exact difference of existing columns (zero new information)"),
 ("ch4/n2o lags and 1-year changes", "-", "-", False, "-", "NOT BUILT: on observed-only series they are short and noisy; on reconstructed series they are interpolation artefacts"),
 ("ch4_was_filled / n2o_was_filled / *_is_synthetic", "reconstruction provenance flags", "-", False, "preserved in output for auditing", "NOT PREDICTORS: kept as metadata to distinguish measured and reconstructed values"),
 ("rolling mean/std for every variable", "-", "-", False, "-", "NOT BUILT: ~145 rows cannot support dozens of near-duplicate predictors"),
 ("co2_growth rolling std", "-", "-", False, "-", "NOT BUILT: no defensible reason beyond co2_std5"),
 ("volcanic/aerosol forcing", "-", "-", False, "-", "EXCLUDED: no complete global series (Phase 1 limitation, affects every result)"),
 ("sea level, sea ice, other temperature datasets", "-", "-", False, "-", "EXCLUDED: consequences/alternative measures of the target -> leakage"),
 ("emissions", "-", "-", False, "-", "EXCLUDED: would need a carbon-cycle model; we use concentrations"),
]
FEATURE_REGISTRY = pd.DataFrame(REGISTRY, columns=["feature", "definition", "scientific_motivation",
                                                   "scenario_capable", "missing_data_limitation", "status_and_use"])
implemented = FEATURE_REGISTRY.loc[FEATURE_REGISTRY["status_and_use"].str.startswith("implemented")
                                   | FEATURE_REGISTRY["status_and_use"].str.startswith("one-year"), "feature"]
unknown_in_table = [f for f in implemented if f not in FEATURE_TABLE.columns and f not in ("sunspots", "sunspots_roll11", "nino34", "ch4", "n2o")]
assert not unknown_in_table, f"registry lists features the table does not have: {unknown_in_table}"
# the registry's scenario-capable list must match the code constant
assert set(FEATURE_REGISTRY.loc[FEATURE_REGISTRY["scenario_capable"], "feature"]) == SCENARIO_SAFE, "registry and SCENARIO_SAFE disagree"
pd.set_option("display.max_colwidth", 80)
display(FEATURE_REGISTRY)

## 6. Redundancy audit: correlation, VIF, and a detrended check

- **Correlation** between two *upward-trending* series is high almost automatically because they share a trend. It does **not** show that one causes the other, and it cannot tell us which feature is useful.
- **VIF** (variance inflation factor) says how well one feature can be rebuilt from the others: `VIF = 1 / (1 - R²)`. Rule of thumb: above ~10 means a feature is largely redundant. Perfectly redundant features have `inf`, and we report that rather than hiding it.
- We compute VIF on the **earlier (training-side) 80% of rows** only, using complete rows (no imputation); constant columns are reported and dropped. The report records the actual years and complete-case sample size for each diagnostic.
- `year_index` is inspected separately because it belongs only to the one-year forecasting experiment; it is not included in scenario-feature diagnostics.
- As a diagnostic, we also correlate **year-to-year changes**. That removes the shared trend; what remains is the short-term relationship. This is for understanding only, not the main scientific task.

In [ ]:
CORR_FEATURES = [f for f in ["co2", "ln_co2", "co2_forcing", "co2_roll5", "co2_roll10", "co2_growth_1y",
                             "co2_growth_roll5", "co2_std5", "co2_forcing_change_5y",
                             "sunspots", "sunspots_roll11", "nino34", "ch4", "n2o"]
                 if f in FEATURE_TABLE.columns]

corr = FEATURE_TABLE[CORR_FEATURES].corr(min_periods=20)     # pairwise-complete; n differs per pair

# Keep the explicit time index out of scenario diagnostics; inspect it only alongside one-year features.
ONE_YEAR_DIAG_FEATURES = [f for f in ["year_index", "temp_at_origin", "temp_lag1", "temp_roll5", "co2_forcing"] if f in FEATURE_TABLE.columns]
year_diag = FEATURE_TABLE[ONE_YEAR_DIAG_FEATURES].corr(min_periods=20)
print("One-year-only diagnostic (includes year_index; not used for scenario selection):")
display(year_diag.round(3))
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(CORR_FEATURES))); ax.set_xticklabels(CORR_FEATURES, rotation=75, ha="right")
ax.set_yticks(range(len(CORR_FEATURES))); ax.set_yticklabels(CORR_FEATURES)
for i in range(len(CORR_FEATURES)):
    for j in range(len(CORR_FEATURES)):
        v = corr.iloc[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation of candidate features (levels share a common trend)")
plt.tight_layout(); plt.show()

def compute_vif(X):
    """VIF from first principles. Handles constant and perfectly collinear columns explicitly."""
    X = X.dropna()
    stds = X.std(ddof=0)
    constant = list(stds[stds == 0].index)
    X = X.drop(columns=constant)
    Z = (X - X.mean()) / X.std(ddof=0)
    out = {}
    for c in Z.columns:
        others = Z.drop(columns=[c])
        if others.shape[1] == 0:
            out[c] = 1.0
            continue
        beta, *_ = np.linalg.lstsq(others.to_numpy(), Z[c].to_numpy(), rcond=None)
        resid = Z[c].to_numpy() - others.to_numpy() @ beta
        r2 = 1.0 - (resid @ resid) / (Z[c].to_numpy() @ Z[c].to_numpy())
        out[c] = np.inf if r2 > 1 - 1e-10 else 1.0 / (1.0 - r2)
    return pd.Series(out, name="VIF"), constant, len(X)

vif_rows = FEATURE_TABLE.sort_values("origin_year")
cut = int(len(vif_rows) * 0.8)
train_side = vif_rows.iloc[:cut]
VIF_SETS = {
    "all_CO2_candidates": [f for f in CO2_FEATURES if f in train_side.columns],
    "base_plus_dynamics": ["co2_forcing", "co2_roll10", "co2_growth_roll5", "co2_std5"],
}
VIF_TABLES = []
for name, cols in VIF_SETS.items():
    cols = [c for c in cols if c in train_side.columns]
    v, constant, n_used = compute_vif(train_side[cols])
    print(f"\nVIF set '{name}': {n_used} complete training-side rows; constant columns dropped: {constant or 'none'}")
    display(v.to_frame())
    complete = train_side[cols].dropna()
    used_years = complete["origin_year"] if "origin_year" in complete.columns else pd.Series(dtype=float)
    VIF_TABLES.append(v.to_frame().assign(feature_set=name, n_rows=n_used,
                                         complete_year_start=int(train_side.loc[complete.index, "origin_year"].min()) if n_used else np.nan,
                                         complete_year_end=int(train_side.loc[complete.index, "origin_year"].max()) if n_used else np.nan).reset_index(names="feature"))
VIF_TABLE = pd.concat(VIF_TABLES, ignore_index=True)

# detrended diagnostic: correlate YEAR-TO-YEAR CHANGES with the next-year temperature change
chg = FEATURE_TABLE[["temp_at_origin"] + [c for c in ["co2_forcing", "sunspots", "nino34", "ch4"] if c in FEATURE_TABLE]].diff()
print("\nCorrelation of 1-year changes with the change in temperature (trend removed; diagnostic only):")
display(chg.corr(min_periods=20)["temp_at_origin"].drop("temp_at_origin").to_frame("corr_with_temp_change"))

## 7. Chronological evaluation engine

**How one fold works (horizon `h`)**
1. List the eligible forecast-origin years (rows where the target and every needed feature exist).
2. The **last ~20%** of those origins become the **hold-out**; the rest are the **development** origins.
3. Development origins are cut into consecutive blocks of 5 years. For each block, the model is trained only on origins `o` with `o + h ≤ first year of the block`. That means every training label was already known at the test block's forecast origin (**purging**). For `h = 0` we use `o ≤ first year − 1` so train and test never share a year.
4. Ridge chooses `alpha` by validating on later blocks *inside the training rows only*.
5. Predictions from all development folds are pooled to compute MAE / RMSE / R². The hold-out gets one separate fit and score.

**Baselines (to beat):**
- **Persistence:** predict that the anomaly `h` years ahead equals today's anomaly. (Not defined for `h = 0`, where it would be the target itself.)
- **Linear time trend:** fit anomaly against calendar year on the training labels and extend the line. Any model that cannot beat a straight line adds little.

R² here is computed relative to the test-period mean; with a handful of points, or a trending test window, it can be very negative or meaningless, so it is blank below 10 test points.

In [ ]:
def assert_feature_rules(features, h):
    for f in features:
        assert not f.startswith("target"), f"target column {f} used as a feature"
        assert f != "origin_year", "origin_year must not be a feature"
        if h == 0:
            assert not f.startswith("temp_"), (f"{f} is derived from temperature; at h=0 the target is that same-year "
                                              "temperature, which would be direct leakage")

def make_pipeline(name, alpha=1.0):
    if name == "linear":
        reg = LinearRegression()
    elif name == "ridge":
        reg = Ridge(alpha=alpha)
    elif name == "forest":
        reg = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=SEED, n_jobs=1)
    else:
        raise ValueError(name)
    # scaler lives INSIDE the pipeline -> fitted on training rows only, every time
    return Pipeline([("scale", StandardScaler()), ("model", reg)])

def make_folds(origins, h):
    """Purged expanding-window folds. Returns [(split, train_origins, test_origins), ...]."""
    origins = sorted(int(o) for o in origins)
    n = len(origins)
    if n < MIN_TRAIN + 2:
        return []
    n_hold = max(1, int(round(HOLDOUT_FRAC * n)))
    dev, hold = origins[: n - n_hold], origins[n - n_hold:]
    gap = max(h, 1)
    train_before = lambda t0: [o for o in origins if o <= t0 - gap]
    folds = []
    for i in range(0, len(dev), TEST_BLOCK):
        test = dev[i:i + TEST_BLOCK]
        train = train_before(test[0])
        if len(train) >= MIN_TRAIN:
            folds.append(("dev", train, test))
    train = train_before(hold[0])
    if len(train) >= MIN_TRAIN:
        folds.append(("holdout", train, hold))
    for split, train, test in folds:           # built-in leakage / overlap guards
        assert set(train).isdisjoint(test), "train/test overlap"
        assert max(train) + gap <= min(test), "a training label would be unknown at the test origin"
    return folds

def choose_alpha(X, y, years, h):
    """Pick Ridge alpha using ONLY the training rows (purged, chronological inner validation)."""
    n = len(years)
    block = max(TEST_BLOCK, n // (INNER_BLOCKS + 3))
    gap = max(h, 1)
    sq = {a: [] for a in ALPHAS}
    used = 0
    for k in range(INNER_BLOCKS, 0, -1):
        s = n - block * k
        if s <= 0:
            continue
        val = np.arange(s, min(s + block, n))
        tr = np.where(years <= years[s] - gap)[0]
        if len(tr) < MIN_INNER_TRAIN or len(val) == 0:
            continue
        used += 1
        for a in ALPHAS:
            m = make_pipeline("ridge", a).fit(X[tr], y[tr])
            sq[a].append((y[val] - m.predict(X[val])) ** 2)
    if used == 0:
        return 1.0
    rmse = {a: float(np.sqrt(np.mean(np.concatenate(v)))) for a, v in sq.items()}
    best = min(rmse.values())
    return max(a for a, r in rmse.items() if r <= best + 1e-12)     # tie -> stronger regularisation

def walk_forward_predictions(rows, h, features, model_name, folds):
    assert_feature_rules(features, h)
    X = rows[list(features)].to_numpy(float)
    y = rows[f"target_h{h}"].to_numpy(float)
    yrs = rows["origin_year"].to_numpy()
    pos = {int(v): i for i, v in enumerate(yrs)}
    parts = []
    for k, (split, train, test) in enumerate(folds):
        tr = np.array([pos[o] for o in train]); te = np.array([pos[o] for o in test])
        alpha = np.nan
        if model_name == "ridge":
            alpha = choose_alpha(X[tr], y[tr], yrs[tr], h)
            model = make_pipeline("ridge", alpha)
        else:
            model = make_pipeline(model_name)
        model.fit(X[tr], y[tr])
        parts.append(pd.DataFrame({"fold": k, "split": split, "origin_year": yrs[te], "y_true": y[te],
                                   "y_pred": model.predict(X[te]), "alpha": alpha, "n_train": len(tr),
                                   "train_origin_start": int(np.min(yrs[tr])), "train_origin_end": int(np.max(yrs[tr])),
                                   "train_target_start": int(np.min(yrs[tr] + h)), "train_target_end": int(np.max(yrs[tr] + h)),
                                   "target_year": yrs[te] + h}))
    return pd.concat(parts, ignore_index=True)

def baseline_predictions(rows, h, kind, folds):
    yrs = rows["origin_year"].to_numpy()
    y = rows[f"target_h{h}"].to_numpy(float)
    ty = rows[f"target_year_h{h}"].to_numpy(float)
    pers = rows["temp_at_origin"].to_numpy(float)
    pos = {int(v): i for i, v in enumerate(yrs)}
    parts = []
    for k, (split, train, test) in enumerate(folds):
        tr = np.array([pos[o] for o in train]); te = np.array([pos[o] for o in test])
        if kind == "trend":
            slope, intercept = np.polyfit(ty[tr], y[tr], 1)
            pred = intercept + slope * ty[te]
        elif kind == "persistence":
            pred = pers[te]
        else:
            raise ValueError(kind)
        parts.append(pd.DataFrame({"fold": k, "split": split, "origin_year": yrs[te], "y_true": y[te],
                                   "y_pred": pred, "alpha": np.nan, "n_train": len(tr),
                                   "train_origin_start": int(np.min(yrs[tr])), "train_origin_end": int(np.max(yrs[tr])),
                                   "train_target_start": int(np.min(yrs[tr] + h)), "train_target_end": int(np.max(yrs[tr] + h)),
                                   "target_year": yrs[te] + h}))
    return pd.concat(parts, ignore_index=True)

def metric_dict(y, p):
    err = y - p
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - float((err ** 2).sum()) / ss_tot if (len(y) >= MIN_R2_N and ss_tot > 0) else np.nan
    return {"MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))), "R2": r2}

def summarise(preds, meta):
    recs = []
    for split, g in preds.groupby("split"):
        recs.append({**meta, "split": split, "n_test": len(g), "n_folds": int(g["fold"].nunique()),
                     "n_train_min": int(g["n_train"].min()), "n_train_max": int(g["n_train"].max()),
                     "test_origin_start": int(g["origin_year"].min()), "test_origin_end": int(g["origin_year"].max()),
                     "test_target_start": int(g["target_year"].min()), "test_target_end": int(g["target_year"].max()),
                     "n_distinct_target_years": int(g["target_year"].nunique()),
                     "train_origin_start_min": int(g["train_origin_start"].min()),
                     "train_origin_end_max": int(g["train_origin_end"].max()),
                     "train_target_start_min": int(g["train_target_start"].min()),
                     "train_target_end_max": int(g["train_target_end"].max()),
                     **metric_dict(g["y_true"].to_numpy(), g["y_pred"].to_numpy()),
                     "alpha_median": float(g["alpha"].median()) if g["alpha"].notna().any() else np.nan})
    return recs

SKIPPED = []

def run_group(group, feature_sets, horizons=None, models=("linear", "ridge")):
    """Evaluate every feature set in `feature_sets` on IDENTICAL rows and folds, plus the two baselines."""
    horizons = EVAL_HORIZONS if horizons is None else horizons
    union = sorted({f for fs in feature_sets.values() for f in fs})
    records = []
    for h in horizons:
        if h not in EVAL_HORIZONS:
            continue
        missing_cols = [f for f in union if f not in FEATURE_TABLE.columns]
        if missing_cols:
            SKIPPED.append({"group": group, "horizon": h, "reason": f"columns not in data: {missing_cols}"})
            continue
        rows = eligible_rows(FEATURE_TABLE, h, union)
        folds = make_folds(rows["origin_year"], h)
        if not folds:
            SKIPPED.append({"group": group, "horizon": h, "reason": f"only {len(rows)} eligible rows - too few for chronological folds"})
            continue
        base_meta = {"group": group, "horizon": h, "n_eligible": len(rows),
                     "eligible_start": int(rows["origin_year"].min()), "eligible_end": int(rows["origin_year"].max())}
        kinds = ["trend"] + ([] if h == 0 else ["persistence"])
        for kind in kinds:
            records += summarise(baseline_predictions(rows, h, kind, folds),
                                 {**base_meta, "feature_set": f"baseline_{kind}", "model": kind, "features": ""})
        for fs_name, feats in feature_sets.items():
            for mname in models:
                preds = walk_forward_predictions(rows, h, feats, mname, folds)
                records += summarise(preds, {**base_meta, "feature_set": fs_name, "model": mname, "features": "+".join(feats)})
    return records
print("Evaluation engine ready.")

## 8. Feature-ablation experiments

Each experiment changes **one thing** and compares on **identical rows and folds**. Feature-selection verdicts use development folds at h=1, 5, and 10 only; h=0 is a contemporaneous mapping, and 15–30-year results remain exploratory because their targets overlap heavily.

| Experiment | Question |
|---|---|
| **A/B** CO₂ transformations | `co2` vs `ln_co2` vs `co2_forcing`. (`ln_co2` and `co2_forcing` differ only by a constant and a scale factor, so after standardisation they are the *same* feature. Expect identical results.) |
| **C** CO₂ dynamics | base `co2_forcing` + **one** of: 5y mean, 10y mean, 1y growth, smoothed growth, **5y std**, 5y forcing change |
| **D** Gases | base + CH₄, base + N₂O (reconstructed annual values; results are retrospective/exploratory because interpolation can use later anchors) |
| **E** Natural drivers | base + sunspots, + 11y sunspot mean, + Niño 3.4 (historical experiments only) |
| **F** Compact set | base + only the add-ons that helped on **development** data |
| **G** One-year experiment | adds temperature history / time index at `h = 1` only |

We do **not** put everything into one model and then guess which feature helped.

In [ ]:
BASE = ["co2_forcing"]
ADDONS_CO2 = ["co2_roll5", "co2_roll10", "co2_growth_1y", "co2_growth_roll5", "co2_std5", "co2_forcing_change_5y"]
ADDONS_GAS = [g for g in GAS_FEATURES if g in FEATURE_TABLE.columns and FEATURE_TABLE[g].notna().sum() > 0]
ADDONS_NAT = [n for n in NATURAL_FEATURES if n in FEATURE_TABLE.columns and FEATURE_TABLE[n].notna().sum() > 0]
ADDON_GROUPS = {f"C_{a}": a for a in ADDONS_CO2}
ADDON_GROUPS.update({f"D_{a}": a for a in ADDONS_GAS})
ADDON_GROUPS.update({f"E_{a}": a for a in ADDONS_NAT})

ALL_RECORDS = []
# A/B: CO2 representation
ALL_RECORDS += run_group("AB_co2_transform", {"co2": ["co2"], "ln_co2": ["ln_co2"], "co2_forcing": ["co2_forcing"]})
# C/D/E: one add-on at a time against the base, on rows where BOTH exist
for gname, feat in ADDON_GROUPS.items():
    ALL_RECORDS += run_group(gname, {"base": BASE, "+" + feat: BASE + [feat]})
RESULTS = pd.DataFrame(ALL_RECORDS)
print("Ablation runs:", len(RESULTS), "result rows | skipped configurations:", len(SKIPPED))
if SKIPPED:
    display(pd.DataFrame(SKIPPED))

In [ ]:
# --- check: ln_co2 and co2_forcing must be indistinguishable for linear models (they are the same feature after scaling)
ab = RESULTS[(RESULTS.group == "AB_co2_transform") & RESULTS.feature_set.isin(["ln_co2", "co2_forcing"])]
piv = ab.pivot_table(index=["horizon", "split", "model"], columns="feature_set", values="RMSE")
assert np.allclose(piv["ln_co2"], piv["co2_forcing"], atol=1e-6), "ln_co2 and co2_forcing should give identical linear-model RMSE"
print("Confirmed: ln_co2 and co2_forcing are the same feature for linear models (identical RMSE).")

# --- development-only verdicts for each add-on (hold-out NOT used)
def addon_verdicts(results):
    rows = []
    for gname, feat in ADDON_GROUPS.items():
        sub = results[(results.group == gname) & (results.model == "ridge") & (results.split == "dev")
                       & (results.horizon.isin(VERDICT_HORIZONS))]
        if sub.empty:
            rows.append({"group": gname, "feature": feat, "verdict": "insufficient data", "horizons_used": 0,
                         "mean_rel_RMSE_change": np.nan, "horizons_improved": 0}); continue
        pv = sub.pivot_table(index="horizon", columns="feature_set", values="RMSE")
        nt = sub[sub.feature_set == "base"].set_index("horizon")["n_test"]
        allowed = nt[(nt >= MIN_TEST_FOR_VERDICT) & nt.index.isin(VERDICT_HORIZONS)].index
        pv = pv.loc[allowed] if len(allowed) else pv.iloc[0:0]
        if pv.empty:
            rows.append({"group": gname, "feature": feat, "verdict": "insufficient data", "horizons_used": 0,
                         "mean_rel_RMSE_change": np.nan, "horizons_improved": 0}); continue
        rel = pv["+" + feat] / pv["base"] - 1.0
        mean_rel, n_imp, n_h = float(rel.mean()), int((rel < 0).sum()), len(rel)
        if mean_rel <= -IMPROVE_MARGIN and n_imp >= math.ceil(n_h / 2):
            verdict = "helped"
        elif mean_rel >= IMPROVE_MARGIN:
            verdict = "hurt"
        else:
            verdict = "no meaningful difference"
        rows.append({"group": gname, "feature": feat, "verdict": verdict, "horizons_used": n_h,
                     "mean_rel_RMSE_change": mean_rel, "horizons_improved": n_imp})
    return pd.DataFrame(rows)

VERDICTS = addon_verdicts(RESULTS)
display(VERDICTS.assign(mean_rel_RMSE_change=(VERDICTS["mean_rel_RMSE_change"] * 100).round(1)).rename(columns={"mean_rel_RMSE_change": "mean_rel_RMSE_change_%"}))
print("Descriptive verdict rule: Ridge, development folds only, horizons", VERDICT_HORIZONS, "with >=", MIN_TEST_FOR_VERDICT,
      "pooled test origins; 'helped' = mean relative RMSE falls >=", f"{IMPROVE_MARGIN:.0%}",
      "and improves at >= half the included horizons. This is NOT a significance test; review horizon-by-horizon results.")
std_row = VERDICTS[VERDICTS.feature == "co2_std5"]
if not std_row.empty:
    r = std_row.iloc[0]
    print(f"\n>>> 5-year CO2 standard deviation: {r.verdict} (mean relative RMSE change {r.mean_rel_RMSE_change:+.1%} over {r.horizons_used} horizons)")

In [ ]:
# --- build the compact feature sets from DEVELOPMENT evidence only
helped = VERDICTS[VERDICTS.verdict == "helped"].copy()  # verdicts use development data at h=1, 5, 10 only
co2_helped = helped[helped.feature.isin(ADDONS_CO2)].sort_values("mean_rel_RMSE_change")
chosen_co2 = [co2_helped.iloc[0].feature] if len(co2_helped) else []          # AT MOST ONE CO2-dynamics add-on (they overlap)
chosen_gas = list(helped[helped.feature.isin(ADDONS_GAS)].feature)
chosen_nat = list(helped[helped.feature.isin(ADDONS_NAT)].feature)

COMPACT_SCENARIO = BASE + chosen_co2 + chosen_gas            # computable from a supplied CO2/gas pathway
COMPACT_HISTORICAL = COMPACT_SCENARIO + chosen_nat           # may also use natural drivers (not scenario-capable)
assert set(COMPACT_SCENARIO) <= SCENARIO_SAFE
assert not any(f.startswith("temp_") or f == "year_index" for f in COMPACT_HISTORICAL)
print("Compact scenario feature set  :", COMPACT_SCENARIO)
print("Compact historical feature set:", COMPACT_HISTORICAL)
if COMPACT_SCENARIO == BASE:
    print("-> No add-on met the evidence rule; the simplest physically motivated model (CO2 forcing only) is the compact set.")

F_SETS = {"base": BASE, "compact_scenario": COMPACT_SCENARIO}
if COMPACT_HISTORICAL != COMPACT_SCENARIO:
    F_SETS["compact_historical"] = COMPACT_HISTORICAL
F_MODELS = ("linear", "ridge") + (("forest",) if RUN_TREE_MODEL else ())
F_RECORDS = run_group("F_compact", F_SETS, models=F_MODELS)
RESULTS = pd.concat([RESULTS, pd.DataFrame(F_RECORDS)], ignore_index=True)

# --- G: the separate ONE-YEAR experiment where temperature history is allowed
G_SETS = {"base": BASE,
          "+temp_now": BASE + ["temp_at_origin"],
          "+temp_now+lag1": BASE + ["temp_at_origin", "temp_lag1"],
          "+temp_now+roll5": BASE + ["temp_at_origin", "temp_roll5"],
          "+year_index": BASE + ["year_index"]}
G_RECORDS = run_group("G_one_year_with_temperature", G_SETS, horizons=[1] if INCLUDE_ONE_YEAR else [])
RESULTS = pd.concat([RESULTS, pd.DataFrame(G_RECORDS)], ignore_index=True)
print("Total result rows:", len(RESULTS))

## 9. Results

Everything below is a *calculated* table from the cells above. Read it with these cautions:
- `n_test` and `n_distinct_target_years` report pooled test origins and distinct target years. These are not independent observations: adjacent origins and long-horizon targets are strongly related. Differences at 15–30 years are exploratory and weak evidence.
- Training/test origin and target-year ranges are reported explicitly. Read the actual number of distinct target years alongside every metric.
- Development metrics come from walk-forward folds; **hold-out** metrics come from one separate block that was never used to choose features or alpha.
- Always compare a model against `baseline_trend` and `baseline_persistence` first.
- **Ridge is not automatically better.** Its shrinkage can under-predict in a steadily rising series, especially when the test years lie beyond the training range. Whichever model wins, the table decides, not the model's reputation.

In [ ]:
def show(group, split, models=None, cols=None):
    t = RESULTS[(RESULTS.group == group) & (RESULTS.split == split)]
    if models is not None:
        t = t[t.model.isin(models)]
    cols = cols or ["horizon", "feature_set", "model", "n_eligible", "n_train_min", "n_train_max", "n_test",
                    "test_origin_start", "test_origin_end", "MAE", "RMSE", "R2", "alpha_median"]
    return t.sort_values(["horizon", "feature_set", "model"])[cols].reset_index(drop=True)

print("=== F: compact feature set vs base vs baselines — DEVELOPMENT folds ===")
display(show("F_compact", "dev").round(3))
print("=== F: compact feature set vs base vs baselines — HOLD-OUT (never used to select features or alpha) ===")
display(show("F_compact", "holdout").round(3))

In [ ]:
# RMSE vs horizon, development folds: do the models beat the two naive baselines?
f_dev = RESULTS[(RESULTS.group == "F_compact") & (RESULTS.split == "dev")]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
curves = [("baseline_trend", "trend"), ("baseline_persistence", "persistence"),
          ("base", "ridge"), ("compact_scenario", "ridge"), ("base", "linear")]
for fs, mdl in curves:
    s = f_dev[(f_dev.feature_set == fs) & (f_dev.model == mdl)].sort_values("horizon")
    if not s.empty:
        axes[0].plot(s["horizon"], s["RMSE"], marker="o", label=f"{fs} / {mdl}")
axes[0].set_xlabel("Horizon (years ahead)"); axes[0].set_ylabel("RMSE (°C)")
axes[0].set_title("Development-fold RMSE by horizon"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
s = f_dev[(f_dev.feature_set == "base") & (f_dev.model == "ridge")].sort_values("horizon")
axes[1].bar(s["horizon"].astype(str), s["n_eligible"], label="eligible origins")
axes[1].bar(s["horizon"].astype(str), s["n_test"], label="pooled dev test points")
axes[1].set_xlabel("Horizon"); axes[1].set_ylabel("count"); axes[1].legend()
axes[1].set_title("Data behind each horizon (fewer for long horizons)")
plt.tight_layout(); plt.show()
print("A model is only useful where its curve is clearly BELOW the trend/persistence baselines, with enough test points.")

In [ ]:
print("=== G: one-year experiment (h=1), temperature history allowed — development folds ===")
g = show("G_one_year_with_temperature", "dev", cols=["horizon", "feature_set", "model", "n_eligible", "n_test", "MAE", "RMSE", "R2"])
display(g.round(3))
print("These temperature-history models cannot be used for scenarios (future temperature is unknown).")

## 10. Scenario feature builder (exploratory, not validated)

**Key design point.** A direct model that only sees *today's* inputs cannot react to a *future* pathway: the features at origin 2025 are the same for every scenario. For scenarios, the pathway must be fed in as features **in the scenario years themselves**. That is why the dashboard-facing model is the `h = 0` (contemporaneous) model: "the CO₂ features of year `Y` → the anomaly of year `Y`". The same code also supports lagged versions (`SCENARIO_DEMO_HORIZON > 0`).

The builder:
- uses the **same formulas** as the historical features (`co2_feature_frame`),
- accepts only a `year` + `co2` (+ optional `ch4`, `n2o`) table, and **refuses** features that would need future temperature, sunspots or El Niño,
- never invents gas values: if the chosen feature set needs CH₄/N₂O, you must supply an explicit assumption.

The example pathways are **illustrative growth-rate assumptions**, configurable below. They are **not** IPCC/SSP scenarios.

In [ ]:
def scenario_features(scenario_df, feature_names, prior_co2=None):
    """Features for a scenario. Uses ONLY the supplied driver columns."""
    if not {"year", "co2"} <= set(scenario_df.columns):
        raise ValueError("scenario table needs at least 'year' and 'co2'")
    bad = sorted(set(feature_names) - SCENARIO_SAFE)
    if bad:
        raise ValueError(f"Features {bad} need information a scenario does not supply "
                         "(future temperature / sunspots / Nino / time index). Remove them or state explicit assumptions.")
    s = scenario_df.sort_values("year").reset_index(drop=True)
    _assert_consecutive(s["year"], "scenario years")
    co2 = s.set_index("year")["co2"].astype(float)
    if prior_co2 is not None:                               # earlier years, only to fill rolling windows
        prior = prior_co2.astype(float).sort_index()
        assert prior.index.max() == co2.index.min() - 1, "prior CO2 must end the year before the scenario starts"
        co2 = pd.concat([prior, co2]).sort_index()
    frame = co2_feature_frame(co2).loc[s["year"]].copy()
    for gas in [g for g in feature_names if g in GAS_FEATURES]:
        if gas not in s.columns:
            raise ValueError(f"Feature set needs {gas}; supply an explicit {gas} assumption column.")
        frame[gas] = s.set_index("year")[gas]
    out = frame[list(feature_names)].copy()
    out.index.name = "origin_year"
    return out.reset_index()

def make_co2_path(first_year, last_year, last_observed_co2, growth_start, growth_end):
    """ILLUSTRATIVE path: CO2 growth (ppm/yr) moves linearly from growth_start to growth_end."""
    years = np.arange(first_year, last_year + 1)
    growth = np.linspace(growth_start, growth_end, len(years))
    return pd.DataFrame({"year": years, "co2": last_observed_co2 + np.cumsum(growth)})

# Editable, documented assumptions (ppm per year). NOT official scenarios.
SCENARIO_ASSUMPTIONS = {"low (illustrative)": (2.4, 0.0),
                        "medium (illustrative)": (2.4, 2.4),
                        "high (illustrative)": (2.4, 4.0)}
SCENARIO_YEARS_AHEAD = 30
SCENARIO_DEMO_HORIZON = 0 if INCLUDE_CONTEMPORANEOUS else 5
print("Scenario builder ready. Demo horizon:", SCENARIO_DEMO_HORIZON)

In [ ]:
# Scenario-capable compact set; the demo only runs if it needs nothing we cannot supply.
needs_gas = [f for f in COMPACT_SCENARIO if f in GAS_FEATURES]
h_demo = SCENARIO_DEMO_HORIZON
DEMO_OK = (h_demo in EVAL_HORIZONS) and not needs_gas
if needs_gas:
    print(f"Demo skipped: the compact scenario set uses {needs_gas}, which would need explicit pathway assumptions.")

if DEMO_OK:
    rows = eligible_rows(FEATURE_TABLE, h_demo, COMPACT_SCENARIO)
    folds = make_folds(rows["origin_year"], h_demo)
    # Out-of-sample residual spread from the walk-forward run (a crude, partial uncertainty band)
    oos = walk_forward_predictions(rows, h_demo, COMPACT_SCENARIO, "ridge", folds)
    resid = oos["y_true"] - oos["y_pred"]
    q_lo, q_hi = np.quantile(resid, [0.05, 0.95])
    Xa, ya, yra = rows[COMPACT_SCENARIO].to_numpy(float), rows[f"target_h{h_demo}"].to_numpy(float), rows["origin_year"].to_numpy()
    alpha_final = choose_alpha(Xa, ya, yra, h_demo)
    final_model = make_pipeline("ridge", alpha_final).fit(Xa, ya)          # fitted on all eligible history
    coef = pd.Series(final_model.named_steps["model"].coef_, index=COMPACT_SCENARIO)
    print(f"Final ridge alpha={alpha_final:g}; standardised coefficients:\n{coef.to_string()}")
    print("Residual 5–95% band from walk-forward predictions: "
          f"[{q_lo:+.2f}, {q_hi:+.2f}] °C (does NOT include pathway, structural, or omitted-forcing uncertainty)")

    last_year = int(annual.loc[annual["co2"].notna(), "year"].max())
    last_co2 = float(annual.set_index("year").loc[last_year, "co2"])
    hist_co2 = annual.set_index("year")["co2"].dropna()
    prior = hist_co2.loc[last_year - 9:last_year]                              # real recent CO2 for rolling windows
    proj = []
    for name, (g0, g1) in SCENARIO_ASSUMPTIONS.items():
        sc = make_co2_path(last_year + 1, last_year + SCENARIO_YEARS_AHEAD, last_co2, g0, g1)
        feats = scenario_features(sc, COMPACT_SCENARIO, prior_co2=prior)
        pred = final_model.predict(feats[COMPACT_SCENARIO].to_numpy(float))
        proj.append(pd.DataFrame({"scenario": name, "origin_year": feats["origin_year"],
                                  "target_year": feats["origin_year"] + h_demo, "co2_assumed": sc["co2"].to_numpy(),
                                  "pred_anomaly": pred, "band_low": pred + q_lo, "band_high": pred + q_hi,
                                  "status": "EXPLORATORY - not validated, not an official projection"}))
    SCENARIO_PROJECTIONS = pd.concat(proj, ignore_index=True)

    # Historical counterfactual: hold CO2 at ~280 ppm for every year (other drivers NOT modelled)
    cf_years = annual["year"].to_numpy()
    cf_df = pd.DataFrame({"year": cf_years, "co2": CO2_PREINDUSTRIAL})
    cf_prior = pd.Series(CO2_PREINDUSTRIAL, index=np.arange(cf_years.min() - 10, cf_years.min()))
    cf_feats = scenario_features(cf_df, COMPACT_SCENARIO, prior_co2=cf_prior)
    cf_pred = final_model.predict(cf_feats[COMPACT_SCENARIO].to_numpy(float))
    fitted = final_model.predict(rows[COMPACT_SCENARIO].to_numpy(float))
    COUNTERFACTUAL = pd.DataFrame({"origin_year": cf_feats["origin_year"], "target_year": cf_feats["origin_year"] + h_demo,
                                   "counterfactual_pred": cf_pred}).merge(
        pd.DataFrame({"origin_year": rows["origin_year"], "fitted_with_actual_co2": fitted}), on="origin_year", how="left").merge(
        pd.DataFrame({"target_year": annual["year"], "observed": annual["temp_anomaly"]}), on="target_year", how="left")
    COUNTERFACTUAL["status"] = "EXPLORATORY - CO2-only approximation; aerosols, volcanoes, ENSO, solar not modelled"

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    axes[0].plot(COUNTERFACTUAL["target_year"], COUNTERFACTUAL["observed"], label="observed", color="k", lw=1)
    axes[0].plot(COUNTERFACTUAL["target_year"], COUNTERFACTUAL["fitted_with_actual_co2"], label="model, actual CO2 (in-sample)")
    axes[0].plot(COUNTERFACTUAL["target_year"], COUNTERFACTUAL["counterfactual_pred"], label="model, CO2 held at 280 ppm")
    axes[0].set_title("CO₂ held at 280 ppm (statistical counterfactual, not causal)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
    axes[0].set_ylabel("Temperature anomaly (°C)")
    for name, g in SCENARIO_PROJECTIONS.groupby("scenario"):
        axes[1].plot(g["target_year"], g["pred_anomaly"], label=name)
        axes[1].fill_between(g["target_year"], g["band_low"], g["band_high"], alpha=0.08)
    axes[1].set_title("Illustrative CO₂ pathways (not IPCC; not validated)"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print("Reminder: h=0 is a same-year regression, not a conventional forecast or causal climate response.")
    print("The shaded band reflects past out-of-sample residual spread only; it omits pathway, model, and omitted-forcing uncertainty.")
    print("The 280 ppm line is a CO₂-held-constant statistical counterfactual, not a no-human-influence world. At ~280 ppm it may be outside/below the training range and rely on extrapolation.")
else:
    SCENARIO_PROJECTIONS = pd.DataFrame(); COUNTERFACTUAL = pd.DataFrame()

## 11. Automated sanity checks

These **fail loudly** (an `AssertionError`) instead of printing a warning. They cover: valid annual index, correct year-based target alignment for every horizon, trailing-only features (proved by truncating the data), finite features, no silent gas filling, no train/test overlap, identical test sets across compared feature sets, correct sample counts, scenario-builder restrictions, explicit target-year reporting, and unchanged input files.

In [ ]:
def check_valid_index():
    assert annual["year"].is_unique and annual["year"].is_monotonic_increasing
    assert annual["year"].diff().dropna().eq(1).all()
    print("OK  valid, sorted, unique, consecutive annual index")

def check_target_alignment():
    temp = annual.set_index("year")["temp_anomaly"]
    ft = FEATURE_TABLE.set_index("origin_year")
    for h in EVAL_HORIZONS:
        for origin in ft.index:
            ty = origin + h
            expected = temp.get(ty, np.nan)
            got = ft.loc[origin, f"target_h{h}"]
            assert (np.isnan(expected) and np.isnan(got)) or got == expected, f"target mismatch h={h} origin={origin}"
            assert ft.loc[origin, f"target_year_h{h}"] == ty
    # explicit worked example: origin 1990 + 5 years -> the real 1995 value (not "5 rows later")
    if 1990 in ft.index and 1995 in temp.index:
        assert ft.loc[1990, "target_h5"] == temp.loc[1995]
        print("OK  worked example: origin 1990, h=5 ->", round(float(temp.loc[1995]), 3), "= observed 1995 anomaly")
    # the last h origins have no target
    for h in EVAL_HORIZONS:
        last_origins = ft.index[-h:] if h > 0 else []
        for o in last_origins:
            assert np.isnan(ft.loc[o, f"target_h{h}"])
    print("OK  year-based target alignment for all horizons; final h origins correctly have no target")

def check_trailing_only(cut_years=None):
    """Features at origin t must be identical when every year after t is deleted."""
    years = annual["year"].to_numpy()
    cut_years = cut_years or [int(years[len(years) // 3]), int(years[len(years) // 2]), int(years[(3 * len(years)) // 4])]
    feat_cols = ALL_FEATURES
    for k in cut_years:
        trunc = build_features(annual[annual["year"] <= k].reset_index(drop=True), CO2_PREHISTORY)
        full = FEATURE_TABLE[FEATURE_TABLE["origin_year"] <= k].reset_index(drop=True)
        pd.testing.assert_frame_equal(trunc[["origin_year"] + feat_cols], full[["origin_year"] + feat_cols],
                                      check_exact=False, rtol=1e-12, atol=1e-12)
    print(f"OK  features use only past/current data (identical after truncating at {cut_years})")

def check_features_finite():
    vals = FEATURE_TABLE[ALL_FEATURES].to_numpy(float)
    assert not np.isinf(vals).any(), "infinite feature values"
    print("OK  no infinite feature values (NaN only where a window/record is genuinely unavailable)")

def check_reconstructed_gases_preserved():
    """Ensure the feature table retains every reconstructed gas value and its provenance flags."""
    annual_by_year = annual.set_index("year")
    features_by_year = FEATURE_TABLE.set_index("origin_year")
    for gas in GAS_FEATURES:
        assert gas in annual_by_year.columns and gas in features_by_year.columns, f"{gas}: missing from input or feature table"
        pd.testing.assert_series_equal(
            features_by_year[gas].rename(gas), annual_by_year[gas].rename(gas),
            check_names=True, check_dtype=False
        )
        for suffix in ("_was_filled", "_is_synthetic"):
            col = gas + suffix
            assert col in annual_by_year.columns and col in features_by_year.columns, f"{col}: provenance metadata missing"
            pd.testing.assert_series_equal(
                features_by_year[col].rename(col), annual_by_year[col].rename(col),
                check_names=True, check_dtype=False
            )
    print("OK  reconstructed CH4/N2O values retained exactly; provenance flags preserved as metadata, not predictors")

def check_folds():
    for h in EVAL_HORIZONS:
        rows = eligible_rows(FEATURE_TABLE, h, ["co2_forcing"])
        for split, tr, te in make_folds(rows["origin_year"], h):
            assert set(tr).isdisjoint(te) and max(tr) + max(h, 1) <= min(te)
            assert list(te) == sorted(te) and list(tr) == sorted(tr)         # chronological, never shuffled
    print("OK  no train/test overlap; chronological folds; training labels known before each test origin")

def check_same_test_sets():
    key = ["group", "horizon", "split"]
    for _, g in RESULTS.groupby(key):
        assert g["n_test"].nunique() == 1, f"different n_test inside {g.iloc[0][key].to_dict()}"
        assert g["test_origin_start"].nunique() == 1 and g["test_origin_end"].nunique() == 1
        assert g["n_eligible"].nunique() == 1
        assert (g["n_distinct_target_years"] <= g["n_test"]).all()
    print("OK  every feature set / model / baseline inside a comparison used identical test origins")

def check_sample_counts():
    for h in EVAL_HORIZONS:
        yrs = set(annual["year"]); temp_ok = set(annual.loc[annual["temp_anomaly"].notna(), "year"])
        expect_origins = {y for y in yrs if (y + h) in temp_ok and y in temp_ok}
        need_cols = ["co2_forcing"]
        exp = [y for y in expect_origins if np.isfinite(FEATURE_TABLE.set_index("origin_year").loc[y, need_cols].to_numpy(float)).all()]
        assert len(eligible_rows(FEATURE_TABLE, h, need_cols)) == len(exp), f"sample count mismatch at h={h}"
    print("OK  eligible sample counts match an independent year-set calculation at every horizon")

def check_no_forbidden_features():
    for name, feats in [("COMPACT_SCENARIO", COMPACT_SCENARIO), ("BASE", BASE)]:
        assert set(feats) <= SCENARIO_SAFE, f"{name} contains non-scenario features"
    for g_name, rec in RESULTS[RESULTS.group.str.startswith(("C_", "D_", "F_", "AB_"))].groupby("group"):
        for f in rec["features"].unique():
            for part in filter(None, f.split("+")):
                assert not part.startswith("temp_") and part != "year_index", f"{part} appears in {g_name}"
    assert all(not f.startswith("target") for f in ALL_FEATURES)
    print("OK  no target / temperature / year-index feature in the scenario-oriented experiments")

def check_scenario_builder():
    # 1) same formulas: rebuilding CO2 features from the historical CO2 column reproduces the historical features
    hist = annual[["year", "co2"]].dropna().reset_index(drop=True)
    pre = CO2_PREHISTORY if CO2_PREHISTORY is not None else None
    rebuilt = scenario_features(hist, CO2_FEATURES, prior_co2=pre)
    ref = FEATURE_TABLE[["origin_year"] + CO2_FEATURES]
    pd.testing.assert_frame_equal(rebuilt.reset_index(drop=True), ref.reset_index(drop=True), check_exact=False, rtol=1e-10, atol=1e-10)
    # 2) outputs contain only requested, scenario-safe columns
    out = scenario_features(hist.tail(30), COMPACT_SCENARIO)
    assert set(out.columns) == {"origin_year", *COMPACT_SCENARIO}
    # 3) forbidden inputs are refused
    for bad in (["temp_lag1"], ["nino34"], ["sunspots"], ["year_index"]):
        try:
            scenario_features(hist.tail(30), bad)
        except ValueError:
            continue
        raise AssertionError(f"scenario builder accepted forbidden feature {bad}")
    print("OK  scenario builder reproduces historical CO2 features exactly and refuses temperature/Nino/sunspot/time features")

def check_inputs_unchanged():
    after = {p.name: sha256(p) for p in sorted(DATA_DIR.glob("*.csv"))}
    assert after == INPUT_HASHES_BEFORE, "an existing processed CSV changed while this notebook ran!"
    print("OK  every existing data/processed/*.csv is byte-for-byte unchanged")

check_valid_index(); check_target_alignment(); check_trailing_only(); check_features_finite()
check_reconstructed_gases_preserved(); check_folds(); check_same_test_sets(); check_sample_counts()
check_no_forbidden_features(); check_scenario_builder(); check_inputs_unchanged()
print("\nALL CHECKS PASSED  (seed =", SEED, ")")

## 12. Save outputs (new folders only)

Nothing in `data/processed/*.csv` is modified. Outputs go to:
- `data/processed/multi_horizon_features/`
- `reports/feature_engineering/multi_horizon/`

In [ ]:
safe_write_csv(FEATURE_TABLE, OUT_DATA_DIR / "climate_annual_features_multi_horizon.csv")
safe_write_csv(FEATURE_REGISTRY, OUT_REPORT_DIR / "feature_registry.csv")
safe_write_csv(SAMPLE_COUNTS, OUT_REPORT_DIR / "sample_counts_by_horizon.csv")
safe_write_csv(VIF_TABLE, OUT_REPORT_DIR / "vif_table.csv")
safe_write_csv(corr.reset_index(names="feature"), OUT_REPORT_DIR / "feature_correlations.csv")
safe_write_csv(RESULTS, OUT_REPORT_DIR / "ablation_results.csv")
safe_write_csv(VERDICTS, OUT_REPORT_DIR / "addon_verdicts_dev_only.csv")
if SKIPPED:
    safe_write_csv(pd.DataFrame(SKIPPED), OUT_REPORT_DIR / "skipped_configurations.csv")
if len(SCENARIO_PROJECTIONS):
    safe_write_csv(SCENARIO_PROJECTIONS, OUT_REPORT_DIR / "scenario_projections_EXPLORATORY.csv")
    safe_write_csv(COUNTERFACTUAL, OUT_REPORT_DIR / "counterfactual_EXPLORATORY.csv")

summary = {
    "notebook": "04_feature_engineering_multi_horizon",
    "input_file": annual_path.name, "using_reconstructed_fallback": USING_RECONSTRUCTED,
    "input_sha256": INPUT_HASHES_BEFORE, "seed": SEED,
    "annual_rows": int(len(annual)), "year_range": [int(annual["year"].min()), int(annual["year"].max())],
    "gap_years": GAP_YEARS, "co2_prehistory": prehistory_msg, "gas_reconstruction_report": GAS_RECONSTRUCTION_REPORT,
    "gas_values_policy": "retain reconstructed CH4/N2O concentrations; provenance flags preserved as metadata, not predictors",
    "horizons_evaluated": EVAL_HORIZONS,
    "config": {"MIN_TRAIN": MIN_TRAIN, "TEST_BLOCK": TEST_BLOCK, "HOLDOUT_FRAC": HOLDOUT_FRAC,
               "ALPHAS": [float(a) for a in ALPHAS], "IMPROVE_MARGIN": IMPROVE_MARGIN,
               "MIN_TEST_FOR_VERDICT": MIN_TEST_FOR_VERDICT, "VERDICT_HORIZONS": VERDICT_HORIZONS},
    "compact_scenario_features": COMPACT_SCENARIO, "compact_historical_features": COMPACT_HISTORICAL,
    "co2_std5_verdict": VERDICTS.loc[VERDICTS.feature == "co2_std5", "verdict"].tolist(),
    "usable_rows_by_horizon": SAMPLE_COUNTS.set_index("horizon")["usable_CO2_forcing"].to_dict(),
    "checks": "all assertions in section 11 passed",
}
safe_write_text(json.dumps(summary, indent=2, default=str), OUT_REPORT_DIR / "run_summary.json")
check_inputs_unchanged()
print("Saved to:", OUT_DATA_DIR, "and", OUT_REPORT_DIR)
display(pd.Series({k: v for k, v in summary.items() if k in ("annual_rows", "year_range", "compact_scenario_features",
                                                               "compact_historical_features", "co2_std5_verdict")}))

## 13. Limitations, interpretation and next steps

**What the results can and cannot say**
- Metrics come from ~145 annual points with a strong shared trend. Good scores can come from the trend alone; always read them against the trend and persistence baselines.
- **15–30-year horizons** have few usable origins and heavily overlapping targets. Treat their scores as exploratory; do not pool them into feature-selection verdicts or read statistical certainty from them. Feature-selection verdicts use development folds at h=1, 5, and 10 only, and are descriptive rather than significance tests.
- **Gas features** use the reconstructed annual CH₄/N₂O concentrations, including filled historical gaps where available. Provenance flags remain in the output for auditing but are never model predictors. Because interpolation can use later anchors, gas-feature experiments are retrospective/exploratory and should not be interpreted as leakage-free real-time forecasts.
- **Omitted forcings** (aerosols, volcanoes), ocean-heat uptake, and feedbacks are absent. The CO₂ coefficient therefore absorbs some of their effect. The 280 ppm curve is a CO₂-held-constant statistical counterfactual, not a causal estimate and not a world without human influence.
- The scenario band is only **past out-of-sample residual spread**. It does not contain pathway, structural, parameter, or omitted-forcing uncertainty. The default h=0 model is a same-year regression, not a conventional forecast.
- **Scenarios are not validated** by historical backtests; the relationship is assumed to stay linear outside the observed CO₂ range.

**Suggested next steps (not done here)**
1. Move `build_features` / `co2_feature_frame` into `src/features/` so training, serving and tests share one implementation, then add a DVC stage (needs team approval; nothing was changed here).
2. Log `ablation_results.csv` and the chosen compact feature set to MLflow in the model-training notebook.
3. Use `run_summary.json` and the file hashes as the feature-stage lineage record for monitoring/drift.

**Decisions that need approval:** see the final report in chat.